# 🔥 Ultra-Ensemble Beast Mode - Google Colab FREE GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/ultra_ensemble_colab.ipynb)

**Target:** TOP 10-100 (43-52% SMAPE)  
**Runtime:** 10-14 hours on FREE T4 GPU  
**Strategy:** 3 Vision + 3 Text + 6 ML Models + Meta-Stacking

---

## 🚀 SETUP (2 minutes):
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save
2. **Upload dataset**: Click folder icon → Upload `sample_train.csv` and `sample_test.csv`
3. **Run all cells**: Runtime → Run all (or Ctrl+F9)
4. **Sit back** and watch the progress!

## ⚡ Features:
- 🆓 **100% FREE** T4 GPU (16GB)
- 🕐 **12-hour limit** (enough for completion)
- 💾 **Auto-download** submissions when done
- 📊 **Real-time progress** tracking

---

In [ ]:
# ============================================
# STEP 1: Enable GPU & Check
# ============================================
# Go to: Runtime → Change runtime type → T4 GPU → Save

import torch
print("🔥 GPU Check:")
if torch.cuda.is_available():
    print(f"   ✅ GPU Enabled: {torch.cuda.get_device_name(0)}")
    print(f"   ✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n🚀 Ready to rock! Continue to next cells.\n")
else:
    print("   ❌ NO GPU! Go to Runtime → Change runtime type → T4 GPU")
    raise Exception("Please enable GPU first!")

In [ ]:
# ============================================
# STEP 2: Install Required Packages (3-5 min)
# ============================================

print("📦 Installing packages... (3-5 minutes)\n")

!pip install -q transformers torch torchvision timm pillow
!pip install -q xgboost lightgbm catboost scikit-learn
!pip install -q pandas numpy requests tqdm scipy

print("\n✅ All packages installed!\n")

In [ ]:
# ============================================
# STEP 3: Import Libraries
# ============================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    CLIPProcessor, CLIPModel
)
import timm
from PIL import Image
import requests
from io import BytesIO
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from tqdm import tqdm
import warnings
import gc
from google.colab import files
import time

warnings.filterwarnings('ignore')
device = torch.device('cuda')

print("✅ Libraries loaded!")
print(f"🔥 Using GPU: {torch.cuda.get_device_name(0)}\n")

In [ ]:
# ============================================
# STEP 4: Upload Dataset Files
# ============================================
# Click the folder icon on left → Upload sample_train.csv and sample_test.csv
# OR run this cell to upload via dialog:

print("📤 Upload your dataset files (sample_train.csv and sample_test.csv):\n")
uploaded = files.upload()

# Verify files
import os
files_list = os.listdir('.')
print(f"\n✅ Files in Colab: {files_list}")

if 'sample_train.csv' in files_list and 'sample_test.csv' in files_list:
    print("\n✅ Dataset uploaded successfully!\n")
else:
    print("\n❌ Missing files! Please upload both CSV files.")

In [ ]:
# ============================================
# STEP 5: Load Data
# ============================================

print("📂 Loading data...\n")

train_df = pd.read_csv('sample_train.csv')
test_df = pd.read_csv('sample_test.csv')

print(f"✅ Train: {train_df.shape}")
print(f"✅ Test: {test_df.shape}")
print(f"\n📊 Target stats:\n{train_df['price'].describe()}\n")

# Show sample
print("📝 Sample data:")
train_df.head(3)

In [ ]:
# ============================================
# STEP 6: Vision Feature Extractor (3 Models)
# ============================================

print("🖼️ Loading 3 vision models...\n")

class VisionExtractor:
    def __init__(self, device):
        self.device = device
        
        # ResNet50
        print("  [1/3] ResNet50...")
        resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.resnet.eval().to(device)
        
        # EfficientNet-B4
        print("  [2/3] EfficientNet-B4...")
        self.efficientnet = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
        self.efficientnet.eval().to(device)
        
        # Vision Transformer
        print("  [3/3] Vision Transformer...")
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.vit.eval().to(device)
        
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        self.success = 0
        self.failed = 0
        print("\n✅ Vision models loaded (4608 features)!\n")
    
    def download_image(self, url):
        try:
            r = requests.get(url, timeout=5)
            img = Image.open(BytesIO(r.content)).convert('RGB')
            self.success += 1
            return img
        except:
            self.failed += 1
            return Image.new('RGB', (224, 224), color='gray')
    
    def extract(self, image_url):
        img = self.download_image(image_url)
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            resnet_feat = self.resnet(img_tensor).squeeze().cpu().numpy()
            eff_feat = self.efficientnet(img_tensor).squeeze().cpu().numpy()
            vit_feat = self.vit(img_tensor).squeeze().cpu().numpy()
        
        return np.concatenate([resnet_feat, eff_feat, vit_feat])

vision_extractor = VisionExtractor(device)

In [ ]:
# ============================================
# STEP 7: Extract Vision Features (3-4 hours)
# ============================================

print("🖼️ Extracting vision features...")
print(f"   Train: {len(train_df)} images")
print(f"   Test: {len(test_df)} images")
print(f"   ⏱️ ETA: 3-4 hours\n")

start_time = time.time()

# Train
print("Processing TRAIN images...")
train_vision_features = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Vision"):
    features = vision_extractor.extract(row['image_link'])
    train_vision_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

train_vision_df = pd.DataFrame(train_vision_features, columns=[f'vision_{i}' for i in range(4608)])
print(f"✅ Train vision: {train_vision_df.shape}")

# Test
vision_extractor.success = 0
vision_extractor.failed = 0
print("\nProcessing TEST images...")
test_vision_features = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Vision"):
    features = vision_extractor.extract(row['image_link'])
    test_vision_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

test_vision_df = pd.DataFrame(test_vision_features, columns=[f'vision_{i}' for i in range(4608)])
print(f"✅ Test vision: {test_vision_df.shape}")

elapsed = (time.time() - start_time) / 3600
print(f"\n⏱️ Vision extraction time: {elapsed:.1f} hours")

# Free memory
del vision_extractor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================
# STEP 8: Text Feature Extractor (3 Models)
# ============================================

print("📝 Loading 3 text models...\n")

class TextExtractor:
    def __init__(self, device):
        self.device = device
        
        # BERT
        print("  [1/3] BERT...")
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-uncased')
        self.bert_model.eval().to(device)
        
        # RoBERTa
        print("  [2/3] RoBERTa...")
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        self.roberta_model = RobertaModel.from_pretrained('roberta-base')
        self.roberta_model.eval().to(device)
        
        # CLIP
        print("  [3/3] CLIP...")
        self.clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model.eval().to(device)
        
        print("\n✅ Text models loaded (2048 features)!\n")
    
    def extract(self, text):
        with torch.no_grad():
            # BERT
            bert_inputs = self.bert_tokenizer(text, return_tensors='pt', truncation=True,
                                             padding='max_length', max_length=128)
            bert_inputs = {k: v.to(self.device) for k, v in bert_inputs.items()}
            bert_feat = self.bert_model(**bert_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # RoBERTa
            roberta_inputs = self.roberta_tokenizer(text, return_tensors='pt', truncation=True,
                                                   padding='max_length', max_length=128)
            roberta_inputs = {k: v.to(self.device) for k, v in roberta_inputs.items()}
            roberta_feat = self.roberta_model(**roberta_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # CLIP
            clip_inputs = self.clip_processor(text=[text], return_tensors='pt', truncation=True,
                                             padding=True, max_length=77)
            clip_inputs = {k: v.to(self.device) for k, v in clip_inputs.items()}
            clip_feat = self.clip_model.get_text_features(**clip_inputs).squeeze().cpu().numpy()
        
        return np.concatenate([bert_feat, roberta_feat, clip_feat])

text_extractor = TextExtractor(device)

In [ ]:
# ============================================
# STEP 9: Extract Text Features (1-2 hours)
# ============================================

print("📝 Extracting text features...")
print(f"   ⏱️ ETA: 1-2 hours\n")

# Train
print("Processing TRAIN texts...")
train_text_features = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    train_text_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

train_text_df = pd.DataFrame(train_text_features, columns=[f'text_{i}' for i in range(2048)])
print(f"✅ Train text: {train_text_df.shape}")

# Test
print("\nProcessing TEST texts...")
test_text_features = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    test_text_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

test_text_df = pd.DataFrame(test_text_features, columns=[f'text_{i}' for i in range(2048)])
print(f"✅ Test text: {test_text_df.shape}\n")

# Free memory
del text_extractor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================
# STEP 10: Advanced Features
# ============================================

print("⚙️ Creating advanced features...\n")

def create_features(df):
    features = pd.DataFrame()
    features['text_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['text_len'] / (features['word_count'] + 1)
    features['upper_ratio'] = df['catalog_content'].str.findall(r'[A-Z]').str.len() / (features['text_len'] + 1)
    features['digit_count'] = df['catalog_content'].str.findall(r'\d').str.len()
    features['has_price'] = df['catalog_content'].str.contains(r'\$|price|cost', case=False).astype(int)
    
    # Brands
    for brand in ['sony','samsung','apple','lg','hp','dell','lenovo','nike','adidas']:
        features[f'brand_{brand}'] = df['catalog_content'].str.lower().str.contains(brand).astype(int)
    
    # Categories
    for cat in ['electronic','clothing','book','home','toy','sport','beauty','food']:
        features[f'cat_{cat}'] = df['catalog_content'].str.lower().str.contains(cat).astype(int)
    
    return features

train_advanced = create_features(train_df)
test_advanced = create_features(test_df)

print(f"✅ Advanced features: {train_advanced.shape[1]}\n")

In [ ]:
# ============================================
# STEP 11: Combine All Features
# ============================================

print("🔗 Combining features...\n")

X_train = pd.concat([train_vision_df, train_text_df, train_advanced], axis=1)
X_test = pd.concat([test_vision_df, test_text_df, test_advanced], axis=1)
y_train = train_df['price'].values

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(f"✅ Final features:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")
print(f"   Vision: 4608")
print(f"   Text: 2048")
print(f"   Advanced: {train_advanced.shape[1]}")
print(f"   TOTAL: {X_train.shape[1]} features 🔥\n")

In [ ]:
# ============================================
# STEP 12: Train 6 ML Models (4-5 hours)
# ============================================

print("🤖 Training 6 models with 5-fold CV...")
print("   ⏱️ ETA: 4-5 hours\n")

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

models = {
    'xgb_1': xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=10,
                             subsample=0.8, colsample_bytree=0.8,
                             tree_method='gpu_hist', random_state=42),
    'xgb_2': xgb.XGBRegressor(n_estimators=1200, learning_rate=0.05, max_depth=8,
                             subsample=0.7, colsample_bytree=0.7,
                             tree_method='gpu_hist', random_state=123),
    'lgb_1': lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, max_depth=10,
                              subsample=0.8, colsample_bytree=0.8,
                              device='gpu', random_state=42),
    'lgb_2': lgb.LGBMRegressor(n_estimators=1200, learning_rate=0.05, max_depth=8,
                              subsample=0.7, colsample_bytree=0.7,
                              device='gpu', random_state=123),
    'cat_1': CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=10,
                              task_type='GPU', verbose=False, random_state=42),
    'cat_2': CatBoostRegressor(iterations=1200, learning_rate=0.05, depth=8,
                              task_type='GPU', verbose=False, random_state=123)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(X_train)) for name in models.keys()}
test_preds = {name: np.zeros(len(X_test)) for name in models.keys()}
cv_scores = {name: [] for name in models.keys()}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/5")
    print(f"{'='*50}")
    
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    for name, model in models.items():
        print(f"  {name}...", end=" ")
        
        if 'xgb' in name:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     early_stopping_rounds=100, verbose=False)
        elif 'lgb' in name:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        else:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                     early_stopping_rounds=100, verbose=False)
        
        val_pred = model.predict(X_val)
        oof_preds[name][val_idx] = val_pred
        test_preds[name] += model.predict(X_test) / 5
        
        score = smape(y_val, val_pred)
        cv_scores[name].append(score)
        print(f"{score:.4f}%")
    
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*50}")
print("✅ TRAINING COMPLETE!")
print(f"{'='*50}\n")

In [ ]:
# ============================================
# STEP 13: Meta-Stacking
# ============================================

print("🔗 Training meta-stacking layer...\n")

meta_train = np.column_stack([oof_preds[name] for name in models.keys()])
meta_test = np.column_stack([test_preds[name] for name in models.keys()])

meta_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.01, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='gpu_hist', random_state=42
)

meta_oof = np.zeros(len(meta_train))
meta_pred = np.zeros(len(meta_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(meta_train)):
    meta_model.fit(meta_train[tr_idx], y_train[tr_idx],
                  eval_set=[(meta_train[val_idx], y_train[val_idx])],
                  early_stopping_rounds=50, verbose=False)
    meta_oof[val_idx] = meta_model.predict(meta_train[val_idx])
    meta_pred += meta_model.predict(meta_test) / 5

meta_cv = smape(y_train, meta_oof)
print(f"✅ Meta-stacking CV: {meta_cv:.4f}%\n")

In [ ]:
# ============================================
# STEP 14: Create Ensemble Strategies
# ============================================

print("📊 Creating 5 ensemble strategies...\n")

# 1. Simple average
simple_avg = sum(test_preds.values()) / len(models)
simple_oof = sum(oof_preds.values()) / len(models)

# 2. Weighted average
weights = np.array([1.0 / np.mean(cv_scores[name]) for name in models.keys()])
weights = weights / weights.sum()
weighted_avg = sum(test_preds[name] * w for name, w in zip(models.keys(), weights))
weighted_oof = sum(oof_preds[name] * w for name, w in zip(models.keys(), weights))

# 3. Best 3 models
sorted_models = sorted(models.keys(), key=lambda x: np.mean(cv_scores[x]))
best_3_avg = sum(test_preds[name] for name in sorted_models[:3]) / 3
best_3_oof = sum(oof_preds[name] for name in sorted_models[:3]) / 3

# 4. Rank averaging
rank_preds = []
for name in models.keys():
    ranks = pd.Series(test_preds[name]).rank(pct=True)
    rank_preds.append(ranks.values)
rank_avg = np.mean(rank_preds, axis=0)
rank_avg_prices = np.percentile(y_train, rank_avg * 100)

# 5. Meta-stacking
meta_final = meta_pred

# Calculate scores
ensemble_scores = {
    'Simple Average': smape(y_train, simple_oof),
    'Weighted Average': smape(y_train, weighted_oof),
    'Best 3 Average': smape(y_train, best_3_oof),
    'Meta-Stacking': meta_cv
}

print("="*60)
print("🏆 FINAL RESULTS")
print("="*60)

for name, score in ensemble_scores.items():
    print(f"{name:<20} {score:>10.4f}% SMAPE")

best_method = min(ensemble_scores, key=ensemble_scores.get)
best_score = ensemble_scores[best_method]

print("\n" + "="*60)
print(f"🥇 BEST: {best_method} → {best_score:.4f}% CV")
print("="*60)
print(f"\n📊 Your Phase 5: 57.900%")
print(f"📊 Improvement: {57.900 - best_score:.2f}%")
print(f"📊 Expected LB: {best_score * 0.98:.2f}% - {best_score * 1.02:.2f}%\n")

if best_score < 46:
    print("🔥🔥🔥 AMAZING! TOP 10-30 POTENTIAL! 🔥🔥🔥")
elif best_score < 50:
    print("⭐⭐⭐ EXCELLENT! TOP 30-100 POTENTIAL! ⭐⭐⭐")
elif best_score < 54:
    print("✅✅✅ VERY GOOD! TOP 100-300 POTENTIAL! ✅✅✅")
else:
    print("✅ GOOD! Beats Phase 5! ✅")

In [ ]:
# ============================================
# STEP 15: Create & Download Submissions
# ============================================

print("💾 Creating submission files...\n")

submissions = {
    'simple_avg': simple_avg,
    'weighted_avg': weighted_avg,
    'best_3_avg': best_3_avg,
    'rank_avg': rank_avg_prices,
    'meta_stacking': meta_final
}

for name, preds in submissions.items():
    df = pd.DataFrame({
        'sample_id': test_df['sample_id'],
        'price': preds
    })
    filename = f'submission_{name}.csv'
    df.to_csv(filename, index=False)
    print(f"   ✅ {filename}")

print("\n" + "="*60)
print("✅ ALL SUBMISSIONS CREATED!")
print("="*60)

print("\n📥 Downloading files to your computer...\n")

# Auto-download all submissions
for name in submissions.keys():
    files.download(f'submission_{name}.csv')

print("\n" + "="*60)
print("🎉 COMPLETE! Check your Downloads folder!")
print("="*60)

print("\n📋 SUBMISSION PRIORITY:")
print("   1️⃣ submission_meta_stacking.csv (BEST)")
print("   2️⃣ submission_best_3_avg.csv")
print("   3️⃣ submission_weighted_avg.csv")

print("\n🚀 Upload to Kaggle and check your rank!")
print("🔥 Good luck reaching TOP 10! 🔥")